# Interfaces e classes abstratas

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Escrever uma interface como contrato e classes que a implementam, e usá-las pelo tipo da interface.
- Escrever uma classe abstrata que faz o trabalho comum e deixa métodos abstratos para cada subclasse.
- Escolher entre uma interface e uma classe abstrata num projeto.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma interface é um contrato
2. Várias interfaces, e métodos default
3. Classes abstratas
4. Interface ou classe abstrata?
5. Uma interface de verdade: Comparable
6. Erros comuns

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Uma interface é um contrato

Uma **interface** lista métodos sem código: ela diz *o que* uma classe sabe fazer, não *como*. Uma classe promete ter esses métodos com `implements`, e precisa escrever cada um deles, como `public`. Aí o código pode usar qualquer uma dessas classes pelo tipo da interface, sem saber qual tem em mãos.

In [ ]:
interface Discount {
    double apply(double price);
}

class Percent implements Discount {
    private final int percent;

    Percent(int percent) {
        this.percent = percent;
    }

    public double apply(double price) {
        return price * (100 - percent) / 100;
    }
}

class Fixed implements Discount {
    public double apply(double price) {
        return Math.max(0, price - 10);
    }
}

void main() {
    List<Discount> offers = List.of(new Percent(20), new Fixed());
    for (Discount d : offers) {
        IO.println(d.getClass().getSimpleName() + ": " + d.apply(30));
    }
}

🧪 **Experimente:** tire o `public` de um `apply` e leia o erro. Depois escreva um terceiro desconto, `HalfPrice`, e ponha na lista.

### ✏️ Exercício: Pontuação de maiúsculas

Termine o `UpperScorer`: a pontuação dele é o número de letras maiúsculas da palavra. O `best` já funciona com qualquer Scorer, então escolhe a palavra com mais maiúsculas.

In [ ]:
interface Scorer {
    int score(String word);
}

class LengthScorer implements Scorer {
    public int score(String word) {
        return word.length();
    }
}

class UpperScorer implements Scorer {
    public int score(String word) {
        // TODO: conte as letras maiúsculas (Character.isUpperCase)
        return 0;
    }
}

String best(List<String> words, Scorer scorer) {
    String best = "";
    for (String w : words) {
        if (best.isEmpty() || scorer.score(w) > scorer.score(best)) {
            best = w;
        }
    }
    return best;
}

void main() {
    check("três maiúsculas", new UpperScorer().score("HeLLo"), 3);
    check("nenhuma maiúscula", new UpperScorer().score("quiet"), 0);
    List<String> words = List.of("banana", "NASA", "JavaScript");
    check("pelo tamanho", best(words, new LengthScorer()), "JavaScript");
    check("pelas maiúsculas", best(words, new UpperScorer()), "NASA");
}

<details>
<summary>Ver uma resposta</summary>

```java
interface Scorer {
    int score(String word);
}

class LengthScorer implements Scorer {
    public int score(String word) {
        return word.length();
    }
}

class UpperScorer implements Scorer {
    public int score(String word) {
        int count = 0;
        for (int i = 0; i < word.length(); i++) {
            if (Character.isUpperCase(word.charAt(i))) {
                count++;
            }
        }
        return count;
    }
}

String best(List<String> words, Scorer scorer) {
    String best = "";
    for (String w : words) {
        if (best.isEmpty() || scorer.score(w) > scorer.score(best)) {
            best = w;
        }
    }
    return best;
}

void main() {
    check("três maiúsculas", new UpperScorer().score("HeLLo"), 3);
    check("nenhuma maiúscula", new UpperScorer().score("quiet"), 0);
    List<String> words = List.of("banana", "NASA", "JavaScript");
    check("pelo tamanho", best(words, new LengthScorer()), "JavaScript");
    check("pelas maiúsculas", best(words, new UpperScorer()), "NASA");
}
```

</details>

## 2. Várias interfaces, e métodos default

Uma classe só pode estender uma classe, mas pode implementar **várias** interfaces, separadas por vírgulas. Uma interface também pode ter um **método `default`**, com código, que toda classe que a implementa ganha de graça e ainda pode sobrescrever.

In [ ]:
interface Named {
    String name();

    default String greeting() {          // código que toda classe Named ganha
        return "Oi, eu sou " + name();
    }
}

interface Priced {
    double price();
}

class Product implements Named, Priced {
    public String name() {
        return "Caderno";
    }

    public double price() {
        return 12.5;
    }
}

void main() {
    Product p = new Product();
    IO.println(p.greeting());
    Priced asPriced = p;                 // o mesmo objeto, visto por outro contrato
    IO.println(asPriced.price());
}

🧪 **Experimente:** sobrescreva `greeting()` em Product. Depois tente `asPriced.name()`: pelo tipo `Priced`, só o `price()` aparece.

## 3. Classes abstratas

Uma **classe abstrata** é um pai que não pode ser criado com `new`. Ao contrário de uma interface, ela pode ter **campos, construtores e métodos prontos**, e marca alguns métodos como `abstract`: sem corpo, e toda subclasse precisa escrevê-los. Um desenho comum é um método pronto que faz o trabalho compartilhado e chama o abstrato para a parte que muda. Os slides acompanham isso.

#### 🎞️ Uma classe abstrata com uma parte em aberto

![p.price() roda o price() escrito uma vez na classe abstrata Ticket, que chama o discount() abstrato; StudentTicket fornece 0.5, então o preço é 50.0; new Ticket(...) é recusado porque Ticket é abstrata](template-method-steps.pt-br.svg)

In [ ]:
interface Priced {
    double price();
}

abstract class Ticket implements Priced {
    final String event;

    Ticket(String event) {
        this.event = event;
    }

    abstract double discount();             // cada tipo de ingresso decide

    public double price() {                 // o trabalho compartilhado, escrito uma vez
        return 100 * (1 - discount());
    }
}

class StudentTicket extends Ticket {
    StudentTicket(String event) {
        super(event);
    }

    double discount() {
        return 0.5;
    }
}

class FullTicket extends Ticket {
    FullTicket(String event) {
        super(event);
    }

    double discount() {
        return 0;
    }
}

void main() {
    List<Ticket> sold = List.of(new StudentTicket("Show"), new FullTicket("Show"));
    for (Ticket t : sold) {
        IO.println(t.event + ": " + t.price());
    }
}

🧪 **Experimente:** tente `new Ticket("Show")` e leia o erro. Depois tire o `discount()` de FullTicket: o compilador exige todos os métodos abstratos.

### ✏️ Exercício: Frete expresso

O `Shipping` cobra 5 mais um preço por quilo, e deixa o `perKg()` abstrato. Faça o `Express` cobrar 4.5 por quilo.

In [ ]:
abstract class Shipping {
    abstract double perKg();

    double cost(double kg) {
        return 5 + perKg() * kg;
    }
}

class Standard extends Shipping {
    double perKg() {
        return 2;
    }
}

class Express extends Shipping {
    double perKg() {
        // TODO: 4.5 por quilo
        return 0;
    }
}

void main() {
    check("padrão, 2 kg", new Standard().cost(2), 9.0);
    check("expresso, 2 kg", new Express().cost(2), 14.0);
    Shipping s = new Express();
    check("expresso pelo Shipping, 0 kg", s.cost(0), 5.0);
}

<details>
<summary>Ver uma resposta</summary>

```java
abstract class Shipping {
    abstract double perKg();

    double cost(double kg) {
        return 5 + perKg() * kg;
    }
}

class Standard extends Shipping {
    double perKg() {
        return 2;
    }
}

class Express extends Shipping {
    double perKg() {
        return 4.5;
    }
}

void main() {
    check("padrão, 2 kg", new Standard().cost(2), 9.0);
    check("expresso, 2 kg", new Express().cost(2), 14.0);
    Shipping s = new Express();
    check("expresso pelo Shipping, 0 kg", s.cost(0), 5.0);
}
```

</details>

## 4. Interface ou classe abstrata?

| | Interface | Classe abstrata |
|---|---|---|
| Uma classe pode ter… | várias (`implements A, B`) | só uma (`extends`) |
| Campos | não (só constantes) | sim |
| Construtor | não | sim |
| Métodos com código | métodos `default` | quaisquer |
| Combina com | uma **capacidade** compartilhada por classes sem parentesco: `Comparable`, `Priced` | uma **família** de classes parentes que dividem estado e código: `Ticket` |

Uma mistura comum: uma interface para o contrato, e uma classe abstrata que a implementa com o código compartilhado, como `Ticket implements Priced` acima.

## 5. Uma interface de verdade: Comparable

A própria biblioteca do Java está cheia de interfaces. A `Comparable<T>` tem um método só, `compareTo`, que devolve um número negativo, zero ou positivo. Uma classe que a implementa ganha uma ordem natural, então o `Collections.sort` e o `TreeSet` conseguem ordenar os objetos dela.

In [ ]:
class Player implements Comparable<Player> {
    final String name;
    final int score;

    Player(String name, int score) {
        this.name = name;
        this.score = score;
    }

    public int compareTo(Player other) {
        return Integer.compare(other.score, score);   // a maior pontuação primeiro
    }

    @Override
    public String toString() {
        return name + " " + score;
    }
}

void main() {
    List<Player> table = new ArrayList<>(List.of(new Player("Ana", 30), new Player("Bia", 75), new Player("Caio", 50)));
    Collections.sort(table);
    IO.println(table);
}

🧪 **Experimente:** troque os argumentos do `Integer.compare` para ordenar da menor pontuação. Depois ponha os jogadores num `TreeSet` em vez de ordenar a lista.

## 6. Erros comuns

- **`new` numa interface ou numa classe abstrata:** nenhuma das duas pode ser criada sozinha; crie uma classe que a implemente ou estenda.
- **Um método faltando:** uma classe que implementa uma interface ou estende uma classe abstrata precisa escrever todos os métodos abstratos, ou ser abstrata também.
- **Esquecer o `public`** num método que implementa um método de interface.
- **`extends` com uma interface, ou `implements` com uma classe:** classes são estendidas; interfaces são implementadas.

In [ ]:
interface Shape {
    double area();
}

class Square implements Shape {
    private final double side;

    Square(double side) {
        this.side = side;
    }

    public double area() {
        return side * side;
    }
}

void main() {
    Shape s = new Square(3);
    IO.println(s.area());
}

🧪 **Experimente:** tente `new Shape()`, depois escreva `class Square extends Shape` e leia cada erro.

## Em resumo

Uma **interface** é um contrato: ela lista métodos sem código, e qualquer classe pode prometer tê-los com `implements`. Uma **classe abstrata** é um pai que não pode ser criado com `new`: ela guarda campos e métodos prontos, e deixa alguns métodos `abstract` para cada subclasse escrever.

```java
interface Priced {
    double price();
}

abstract class Ticket implements Priced {
    final String event;

Ticket(String event) {
        this.event = event;
    }

abstract double discount();

public double price() {
        return 100 * (1 - discount());
    }
}

class StudentTicket extends Ticket {
    StudentTicket(String event) {
        super(event);
    }

double discount() {
        return 0.5;
    }
}

void main() {
    Priced p = new StudentTicket("Show");
    IO.println(p.price());
}
```

Use uma **interface** quando classes sem parentesco compartilham uma capacidade (uma classe pode implementar várias). Use uma **classe abstrata** quando classes parentes compartilham estado e código (uma classe estende só uma).

**Cuidado:** os métodos que implementam uma interface precisam ser `public`, porque os métodos de interface são públicos.

## Leituras

- [Abstract Methods and Classes](https://dev.java/learn/language/oop/inheritance/abstract-classes/) (dev.java (Oracle), 7 min): O que torna um método ou uma classe abstratos, depois a seção "Abstract Classes Compared to Interfaces", que diz quando escolher cada um, e o exemplo do GraphicObject.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Animais (interfaces)** · Médio
- **Notificações (classe abstrata)** · Médio